# Churn-Radar — Análise Exploratória, Interpretabilidade e Negócio

Telecom churn (`barun2104/telecom-churn`). Comparação de 3 modelos:
**Logistic Regression**, **Random Forest** e **XGBoost**.

> O download dos dados via `kagglehub` deve rodar **localmente** com um `.env` válido
> (acesso ao Kaggle costuma estar bloqueado em ambientes de nuvem).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src.data import load_data
from src.features import make_splits
from src.train import build_models
from src.evaluate import compare_models, confusion
from src.business import roi_report, optimize_threshold
from src import config

## 1. Carga dos dados

In [ ]:
df = load_data()
print(df.shape)
df.head()

## 2. EDA — distribuição do alvo e correlações

In [ ]:
print(df[config.TARGET_COL].value_counts(normalize=True))
sns.countplot(x=config.TARGET_COL, data=df)
plt.title('Distribuição de Churn')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm', center=0)
plt.title('Mapa de correlação')
plt.show()

## 3. Treino dos 3 modelos e comparação

In [ ]:
X_train, X_test, y_train, y_test = make_splits(df)
models = build_models(y_train)
for name, m in models.items():
    m.fit(X_train, y_train)
results = compare_models(models, X_test, y_test)
results

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for name, m in models.items():
    RocCurveDisplay.from_estimator(m, X_test, y_test, ax=ax[0], name=name)
    PrecisionRecallDisplay.from_estimator(m, X_test, y_test, ax=ax[1], name=name)
ax[0].set_title('ROC'); ax[1].set_title('Precision-Recall')
plt.show()

## 4. Interpretabilidade — SHAP
TreeExplainer para os modelos de árvore; coeficientes para a Logistic Regression.

In [ ]:
xgb = models['XGBoost']
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

In [ ]:
import numpy as np
logreg = models['LogisticRegression']
coefs = pd.Series(logreg.named_steps['clf'].coef_[0], index=X_train.columns)
coefs.sort_values().plot(kind='barh', figsize=(8, 6), title='Coeficientes — Logistic Regression')
plt.show()

## 5. Métricas de negócio — ROI da estratégia de retenção
Otimiza o threshold para maximizar o lucro líquido esperado da campanha de retenção
(parâmetros de custo/CLV em `src/config.py`).

In [ ]:
proba = xgb.predict_proba(X_test)[:, 1]
report = roi_report(y_test.values, proba)
print(report)
grid = optimize_threshold(y_test.values, proba)
grid_sorted = grid.sort_values('threshold')
plt.plot(grid_sorted['threshold'], grid_sorted['profit'])
plt.axvline(report['best_threshold'], color='r', ls='--', label='threshold ótimo')
plt.xlabel('threshold'); plt.ylabel('lucro esperado'); plt.legend()
plt.title('Lucro da campanha de retenção vs. threshold')
plt.show()